# Домашнее задание: Система NER с использованием YandexGPT

**Автор:** Горковенко Максим
- Разработка системы извлечения именованных сущностей (NER) из юридических документов с использованием API YandexGPT.

## Структура отчета

1. Настройка окружения
2. Описание промптов
3. Примеры успешного извлечения
4. Анализ ошибок и edge cases
5. Сравнение с классическими ML-моделями


---
## 1. Настройка окружения

In [1]:
# Импорт необходимых библиотек
import os
import sys
import json
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# Добавляем путь к модулям
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

# Импорт модуля подгрузки датасета из HF
from src.ner.dataset_utils import LegalDataset

# Импорт NER блока с правилами
from src.ner.extractor import NERExtractor

# Импорт блока с промптами
from src.ner.prompts import SYSTEM_PROMPT

print("Все модули успешно импортированы")

Все модули успешно импортированы


## 2. Описание промптов

### Системный промпт

In [3]:
# Вывод системного промпта (первые 2000 символов)
print("Системный промпт (первые 2000 символов):")
print("=" * 60)
print(SYSTEM_PROMPT[:2000])
print("...")
print(f"\nОбщая длина промпта: {len(SYSTEM_PROMPT)} символов")

Системный промпт (первые 2000 символов):
Ты - строгий и точный AI-помощник юриста. Твоя задача - извлекать факты из юридических документов без искажений.

ВНИМАНИЕ: возвращай ответ ТОЛЬКО в формате валидного JSON-объекта, без сопроводительного текста, маркдаун-тегов или дополнительных комментариев.

Извлекай следующие типы сущностей:

1. **Реквизиты сторон**:
   - Названия компаний/организаций
   - ИНН (идентификационные номера налогоплательщика)
   - КПП (при наличии)

2. **Даты**:
   - Дата подписания документа
   - Сроки действия договора
   - Конкретные даты в тексте

3. **Суммы**:
   - Все финансовые значения
   - Указывай валюту (RUB, USD, EUR и т.д.)
   - Если валюта не указана, предполагай RUB

4. **Сроки обязательств**:
   - Срок действия договора
   - Сроки оплаты
   - Сроки исполнения обязательств

Пример структуры ответа (используй эту схему):

{
  "companies": [
    {
      "name": "Полное название компании или null",
      "inn": "ИНН или null",
      "kpp": "КПП или null

### Основные параметры модели

- **Модель:** yandexgpt-lite
- **Temperature:** 0.2 (низкая для минимизации галлюцинаций)
- **Max Tokens:** 2000
- **Формат ответа:** JSON

---
## 3. Примеры успешного извлечения

### Пример 1: Документ с полной информацией

In [4]:
# Инициализация экстрактора
extractor = NERExtractor()

# Загрузка датасета
dataset = LegalDataset()
dataset.load("train")

# Получение примера
sample = dataset.get_sample("train", index=5)

print("Исходный текст (первые 500 символов):")
print("=" * 60)
print(sample['text'][:500])
print("...")
print(f"\nОбщая длина: {len(sample['text'])} символов")

Загрузка датасета TryDotAtwo/russian-legal-ner...


Датасет успешно загружен.
Загружено 73734 документов для обучения
Исходный текст (первые 500 символов):


	       	Решение по уголовному делу
      	

дело № 1-110-04/2013                                                                            <ОБЕЗЛИЧЕНО>                                             
П О С Т А Н О В Л Е Н И Е
гор. Волгоград «18» февраля 2013г.
мировой судья судебного участка № 110 Волгоградской области (Тракторозаводский район гор. Волгограда) Коновалова С.А.с участием государственного обвинителя                        Бреусовой И.А.,
подсудимого Тасмаджиева Ю.С.,защитника  
...

Общая длина: 8823 символов


In [5]:
# Извлечение сущностей
result = extractor.extract_from_text(
    sample['text'],
    document_id="example_001",
    temperature=0.2
)

print(f"Успех: {result.success}")
print(f"Компании: {result.metrics['total_companies']}")
print(f"Даты: {result.metrics['total_dates']}")
print(f"Суммы: {result.metrics['total_amounts']}")
print(f"Сроки: {result.metrics['total_terms']}")

Успех: True
Компании: 2
Даты: 5
Суммы: 21
Сроки: 1


In [6]:
# Вывод извлеченных данных в красивом формате
print("Извлеченные компании:")
print("-" * 60)

for comp in result.extracted_data.get("companies", []):
    print(f"  Название: {comp.get('name', 'N/A')}")
    print(f"  ИНН: {comp.get('inn', 'N/A')}")
    print(f"  КПП: {comp.get('kpp', 'N/A')}")
    print()

Извлеченные компании:
------------------------------------------------------------
  Название: ЗАО «<ОБЕЗЛИЧЕНО>»
  ИНН: None
  КПП: None

  Название: ЗАО «Группа Тауэр Телеком»
  ИНН: None
  КПП: None



In [7]:
print("Извлеченные суммы:")
print("-" * 60)

for amt in result.extracted_data.get("amounts", []):
    print(f"  {amt.get('description', 'N/A')}: {amt.get('value', 0)} {amt.get('currency', 'RUB')}")

Извлеченные суммы:
------------------------------------------------------------
  Стоимость контрафактного экземпляра авторского произведения: 82500 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 16500 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 5000 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 25000 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 25000 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 25000 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 5000 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 5000 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 7000 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 9900 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 5000 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 5000 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 21450 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 21450 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 21450 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 16150 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 4950 RUB
  Стоимость библиотеки «<ОБЕЗЛИЧЕНО>»: 4950 RUB
  Стоимость библиотеки 

In [8]:
print("Извлеченные сроки:")
print("-" * 60)

for term in result.extracted_data.get("terms", []):
    print(f"  {term.get('description', 'N/A')}: {term.get('value')} {term.get('unit', 'N/A')}")

Извлеченные сроки:
------------------------------------------------------------
  Срок обжалования постановления: 10 days


---
## 4. Анализ ошибок и Edge Cases

### Тест на галлюцинации (документы без сумм)

In [9]:
# Тестирование на отсутствие данных
print("Тестирование документов без сумм...")

edge_results = extractor.extract_specific_cases(dataset, n_no_amount=5)

# Статистика
total = len(edge_results)
hallucinations = sum(1 for r in edge_results if r.get("has_hallucination"))

print("\n" + "=" * 60)
print("Результаты edge case тестов:")
print(f"Всего документов: {total}")
print(f"С галлюцинациями: {hallucinations}")
print(f"Без галлюцинаций: {total - hallucinations}")
print(f"Процент корректности: {(1 - hallucinations/total) * 100:.1f}%")

Тестирование документов без сумм...

Тест 1: Документы без сумм (проверка на галлюцинации)...
  Документ 1: галлюцинаций = True
  Документ 2: галлюцинаций = True
  Документ 3: галлюцинаций = True
  Документ 4: галлюцинаций = True
  Документ 5: галлюцинаций = True

Результаты edge case тестов:
Всего документов: 5
С галлюцинациями: 5
Без галлюцинаций: 0
Процент корректности: 0.0%


### Тест на длинные документы

In [10]:
# Тест с длинными документами
print("Тестирование длинных документов...")

samples = dataset.get_random_samples("train", 10)

long_docs = [s for s in samples if len(s["text"]) > 5000]
print(f"Найдено длинных документов: {len(long_docs)} из 10")

if long_docs:
    result = extractor.extract_from_text(
        long_docs[0]["text"],
        document_id="long_doc_test",
        temperature=0.2
    )
    print(f"\nРезультат:")
    print(f"  Использовано чанков: {result.chunks_used}")
    print(f"  Успех: {result.success}")
    print(f"  Компании: {result.metrics['total_companies']}")
    print(f"  Суммы: {result.metrics['total_amounts']}")

Тестирование длинных документов...
Найдено длинных документов: 2 из 10

Результат:
  Использовано чанков: 1
  Успех: True
  Компании: 1
  Суммы: 2


### Обработка JSON ошибок

In [11]:
# Демонстрация обработки ошибок парсинга
from src.ner.parser import NERParser

parser = NERParser()

# Тест с невалидным JSON
invalid_json = """
Вот мой ответ в JSON формате:
```json
{
  "companies": [{"name": "Test"}]
}
"""

result = parser.extract_entities(invalid_json)
print("Запрос с Markdown-тегами и валидным JSON")
print(f"Успех: {result.get('error') is None}")
print(f"Компании извлечены: {len(result.get('companies', []))}")

Запрос с Markdown-тегами и валидным JSON
Успех: True
Компании извлечены: 1


---
## 5. Сравнение с классическими ML-моделями

p.s.: Сравниваю со своим опытом в NER - в основном использовал регулярки. Для сложных сценариев использовал библиотеку Natasha. Возни ОЧНЬ много на создание правил, но и скорость работы на больших массивах огромна. Применял для работы с номенклатурой - названия позиций, артикулы, и т.д. Задачи вроде "Маркетинговый отдел решил поменять порядок слов в каждом из 1М названий. Там ручной ввод - нужно навести порядок".

### Плюсы LLM-подхода (YandexGPT):
1. **Не требует обучений** - модель уже обучена на больших данных и общие (частотные) терины извлекает хорошо. 
2. **Гибкость промптов** - легко добавить новые типы сущностей через промпт (но только то, что знают ЛЛМ, без проф-перекосов)
3. **Работает с редкими случаями** - понимает неочевидные паттерны. Написание такой регулярки очень долго и дорого.
4. **Меньше предобработки** - не нужно нормализовать текст вручную.
5. **Контекстуальное понимание** - учитывает контекст при извлечении (весь документ а не только строку)

### Минусы LLM-подхода:

1. **Стоимость** - API YandexGPT платный. Чем данных больше, тем дороже.
2. **Скорость** - Медленнее, чем классические подходы на регулярках.
3. **Галлюцинации** - модель может выдумывать данные. Самый болезненный минус в работе с корп-данными.
4. **Нестабильность формата** - иногда возвращает невалидный JSON.
5. **Негибкость** - по опыту ЛЛМ плох в узко-проф-терминах. Точность на электрооборудовании сильно падает, например АВДТ или АВТ различает плохо. Регулярки все еще лучше в таких задачах.

Мы в компании используем в ЭДО и регулярки и LLM - в зависимости от задачи.

### Плюсы классических ML (Natasha/Spacy):

1. **Бесплатно** - open-source модели.
2. **Быстро** - локальное выполнение, нет сетевых задержек.
3. **Стабильность** - детерминированный результат.
4. **Гибкость** - извлекает все, что ты сумеешь описать.

### Минусы классических ML:

1. **Требует выработки правил** - нужны сценарии\правила\данные для обучения.
2. **Меньше гибкости** - сложно добавить новые типы сущностей.
3. **Предобработка** - требуется нормализация текста.
4. **Не понимает контекст** - работают на уровне n-грамм (и то черевато комбинаторным взрывом)


---
## 6. Заключение

### Выводы:

1. **YandexGPT готов к production-задачам NER** при правильной настройке промптов и адекватной задаче.
2. **Требуется обработка edge cases** - много проверок как минимум на типы\диапазоны и т.д. В идеале human-in-the-loop.
3. **Нужен парсинг ответов** - модель иногда возвращает невалидный JSON. Новые модели страдают этим реже.
4. **Chunking критичен** - для длинных документов необходим. У меня простая реализация, но это либо боль либо дорого. 
5. **Низкая температура (0.1-0.3)** обязательна для минимизации галлюцинаций.

### Код доступен в репозитории:

- `src/ner/` - основные модули
- `src/ner/test_ner.py` - скрипт тестирования
- `.env.example` - шаблон для API ключей
